Dependencies

In [1]:
%%bash
pip install -q "qdrant-client[fastembed]>=1.14.2" sentence-transformers transformers accelerate pydantic gradio pandas numpy tqdm

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


In [2]:
import os, glob, shutil, subprocess, time, urllib.request, re
import pandas as pd
from qdrant_client import QdrantClient, models

WORK_STORAGE = "/kaggle/working/qdrant_storage"
WORK_BIN = "/kaggle/working/qdrant_bin/qdrant"
WORK_ARTIFACTS = "/kaggle/working/artifacts"

# If running in a fresh Notebook 2 with Notebook 1 attached in /kaggle/input/, copy writable storage to /kaggle/working
if not os.path.exists(WORK_STORAGE):
    in_storage = glob.glob("/kaggle/input/**/qdrant_storage", recursive=True)
    in_bin = glob.glob("/kaggle/input/**/qdrant_bin/qdrant", recursive=True)
    in_art = glob.glob("/kaggle/input/**/artifacts", recursive=True)
    assert in_storage and in_art, "Cannot find qdrant_storage/artifacts in /kaggle/working or /kaggle/input!"
    print("Copying Qdrant storage from input to writable /kaggle/working...")
    shutil.copytree(in_storage[0], WORK_STORAGE)
    shutil.copytree(in_art[0], WORK_ARTIFACTS, dirs_exist_ok=True)
    os.makedirs("/kaggle/working/qdrant_bin", exist_ok=True)
    if in_bin:
        shutil.copy2(in_bin[0], WORK_BIN)
    else:
        subprocess.run(
            "wget -q https://github.com/qdrant/qdrant/releases/latest/download/qdrant-x86_64-unknown-linux-musl.tar.gz -O /tmp/q.tar.gz && tar -xzf /tmp/q.tar.gz -C /kaggle/working/qdrant_bin",
            shell=True, check=True,
        )
    os.chmod(WORK_BIN, 0o755)

with open("/kaggle/working/qdrant_config.yaml", "w") as f:
    f.write("storage:\n  storage_path: /kaggle/working/qdrant_storage\nservice:\n  http_port: 6333\n  grpc_port: 6334\ntelemetry_disabled: true\n")

Copying Qdrant storage from input to writable /kaggle/working...


In [3]:
# Start Qdrant if not already running
def qdrant_alive():
    try:
        with urllib.request.urlopen("http://localhost:6333/healthz", timeout=1) as r:
            return r.status == 200
    except Exception:
        return False

if not qdrant_alive():
    qdrant_proc = subprocess.Popen(
        [WORK_BIN, "--config-path", "/kaggle/working/qdrant_config.yaml"],
        stdout=open("/tmp/qdrant2.log", "w"),
        stderr=subprocess.STDOUT,
    )
    for _ in range(30):
        if qdrant_alive():
            break
        time.sleep(1)
    assert qdrant_alive(), "Qdrant failed to start. Check /tmp/qdrant2.log"

client = QdrantClient(url="http://localhost:6333", prefer_grpc=True, timeout=60)
COLLECTION_NAME = "recipes"
print("Qdrant UP! Points:", client.get_collection(COLLECTION_NAME).points_count)

# --- Title-Consistency Safety Patch (catches omitted ingredients like 'on Toast') ---
df_clean = pd.read_parquet(f"{WORK_ARTIFACTS}/recipes_clean.parquet")

TITLE_GLUTEN_RE = re.compile(r"\b(toast|bread|sandwich|bagel|pasta|spaghetti|macaroni|pizza|noodle|noodles|ramen|udon|tortilla|pita|biscuit|waffle|pancake|muffin|cake|brownie|cookie|pie|beer|ale|croissant|brioche|couscous|bulgur|seitan|soy sauce)\b", re.I)
TITLE_GF_EXEMPT_RE = re.compile(r"\b(gluten[- ]free|flourless|keto|cauliflower|zucchini|rice noodle|rice paper|corn tortilla)\b", re.I)
TITLE_HALAL_RE = re.compile(r"\b(pork|bacon|ham|prosciutto|pancetta|pepperoni|wine|beer|bourbon|whiskey|rum|vodka|brandy|champagne|sake|mirin)\b", re.I)
TITLE_HALAL_EXEMPT_RE = re.compile(r"\b(turkey bacon|beef bacon|non[- ]alcoholic|ginger ale|root beer|vinegar)\b", re.I)
TITLE_VEGAN_RE = re.compile(r"\b(chicken|beef|pork|turkey|lamb|salmon|tuna|shrimp|bacon|ham|egg|eggs|cheese|feta|cheddar|parmesan|mozzarella|butter|cream|milk|yogurt|honey|mayo)\b", re.I)
TITLE_VEGAN_EXEMPT_RE = re.compile(r"\b(vegan|plant[- ]based|dairy[- ]free|coconut milk|almond milk|soy milk|oat milk|peanut butter|butternut)\b", re.I)
TITLE_NUT_RE = re.compile(r"\b(peanut|almond|walnut|pecan|cashew|pistachio|hazelnut|macadamia|pesto|nutella|praline)\b", re.I)

patched = 0
for row in df_clean.itertuples(index=False):
    title = str(row.Name)
    tags = list(row.diet_tags)
    conf = dict(row.diet_confidence)
    changed = False

    if "gluten_free" in tags and TITLE_GLUTEN_RE.search(title) and not TITLE_GF_EXEMPT_RE.search(title):
        tags.remove("gluten_free"); conf["gluten_free"] = "unknown"; changed = True
    if "halal_friendly" in tags and TITLE_HALAL_RE.search(title) and not TITLE_HALAL_EXEMPT_RE.search(title):
        tags.remove("halal_friendly"); conf["halal_friendly"] = "unknown"; changed = True
    if "vegan" in tags and TITLE_VEGAN_RE.search(title) and not TITLE_VEGAN_EXEMPT_RE.search(title):
        tags.remove("vegan"); conf["vegan"] = "unknown"; changed = True
    if "nut_free" in tags and TITLE_NUT_RE.search(title):
        tags.remove("nut_free"); conf["nut_free"] = "unknown"; changed = True

    if changed:
        client.set_payload(
            collection_name=COLLECTION_NAME,
            payload={"diet_tags": tags, "diet_confidence": conf},
            points=[int(row.RecipeId)],
            wait=False,
        )
        patched += 1

print(f"Title-consistency safety patch downgraded {patched:,} ambiguous/incomplete recipes.")

Qdrant UP! Points: 60000
Title-consistency safety patch downgraded 4,890 ambiguous/incomplete recipes.


Loading encoders and LLM

In [4]:
import json, torch
from sentence_transformers import SentenceTransformer, CrossEncoder
from fastembed import SparseTextEmbedding
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running inference on:", device)

with open(f"{WORK_ARTIFACTS}/taxonomy.json") as f:
    TAXONOMY = json.load(f)
SYNONYMS = TAXONOMY["synonyms"]
KNOWN_VOCAB = set(TAXONOMY["known_vocab"])

dense_model = SentenceTransformer("BAAI/bge-small-en-v1.5", device=device)
sparse_model = SparseTextEmbedding(model_name="Qdrant/bm25")
reranker_model = CrossEncoder("BAAI/bge-reranker-base", device=device, max_length=256)

# Small open-weights GPU LLM for complex query fallback (~6 GB VRAM fp16)
LLM_NAME = "Qwen/Qwen2.5-3B-Instruct"
llm_tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
)
llm_model.eval()
print("All models loaded successfully!")

Running inference on: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

... (28 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

All models loaded successfully!


Hybrid Rules + Pydantic-Validated LLM Fallback Query Parser

In [5]:
import re, json
from typing import Optional, Literal
from pydantic import BaseModel, Field, ValidationError

ALLERGEN_EXPANSIONS = {
    "dairy": ["milk", "butter", "cream", "heavy cream", "sour cream", "cheese", "cheddar", "parmesan", "mozzarella", "feta", "cream cheese", "yogurt", "greek yogurt", "ghee", "buttermilk"],
    "lactose": ["milk", "butter", "cream", "heavy cream", "sour cream", "cheese", "cheddar", "parmesan", "mozzarella", "feta", "cream cheese", "yogurt", "ice cream"],
    "eggs": ["egg"],
    "egg": ["egg"],
    "nuts": ["peanuts", "peanut butter", "almonds", "walnuts", "pecans", "cashews", "pistachios", "hazelnuts", "pine nuts", "pesto"],
    "peanuts": ["peanuts", "peanut butter", "peanut oil"],
    "peanut": ["peanuts", "peanut butter", "peanut oil"],
    "shellfish": ["shrimp", "crab", "lobster", "prawn", "scallop", "clam", "mussel", "oyster"],
    "seafood": ["fish", "salmon", "tuna", "cod", "shrimp", "crab", "lobster", "anchovy"],
    "soy": ["soy sauce", "tofu", "soy milk", "edamame", "tempeh", "miso"],
}

# Build multi-word -> canonical phrase list sorted longest-first for greedy extraction
NORM_PHRASES = sorted(set(SYNONYMS.keys()) | KNOWN_VOCAB, key=len, reverse=True)
SINGLE_TOKENS = {p for p in NORM_PHRASES if " " not in p and len(p) > 2}
MULTI_PHRASES = [p for p in NORM_PHRASES if " " in p]

In [6]:
def normalise_query_term(term: str) -> str:
    t = term.lower().strip()
    if t in SYNONYMS:
        return SYNONYMS[t]
    if t.endswith("s") and t[:-1] in SYNONYMS:
        return SYNONYMS[t[:-1]]
    if t.endswith("s") and t[:-1] in KNOWN_VOCAB:
        return t[:-1]
    return t

class ParsedQuery(BaseModel):
    include_ingredients: list[str] = Field(default_factory=list)
    exclude_ingredients: list[str] = Field(default_factory=list)
    exclude_allergens: list[str] = Field(default_factory=list)
    diet_tags: list[Literal["gluten_free", "halal_friendly", "vegan", "nut_free"]] = Field(default_factory=list)
    max_minutes: Optional[int] = Field(default=None, ge=1, le=480)
    semantic_query: str = ""
    parser_mode: Literal["rules", "llm_fallback"] = "rules"

In [7]:
DIET_PATTERNS = [
    ("gluten_free", re.compile(r"\b(gluten[- ]?free|celiac|no gluten|\bgf\b)\b", re.I)),
    ("halal_friendly", re.compile(r"\b(halal|no pork|pork[- ]?free)\b", re.I)),
    ("vegan", re.compile(r"\b(vegan|plant[- ]?based)\b", re.I)),
    ("nut_free", re.compile(r"\b(nut[- ]?free|no nuts|peanut[- ]?free|tree[- ]?nut[- ]?free)\b", re.I)),
]

TIME_RE = re.compile(r"(?:under|less than|<=|within|in|max|at most)?\s*(\d{1,3})\s*(?:min|mins|minute|minutes)\b", re.I)
NEG_RE = re.compile(r"\b(?:no|without|exclude|excluding|avoid|allergic to|free of)\s+([a-z][a-z\s]{1,25}?)(?=,|\.|;|\band\b|\bunder\b|\bwith\b|$)", re.I)
COMPLEX_HINT_RE = re.compile(r"\b(instead of|except|substitute|something like|craving|kid[- ]friendly|high protein|low carb|not too)\b", re.I)

In [8]:
def extract_ingredients_from_text(text: str, blocked: set[str]) -> list[str]:
    s = f" {text.lower()} "
    found = []
    for ph in MULTI_PHRASES:
        if f" {ph} " in s:
            canon = normalise_query_term(ph)
            if canon not in blocked and canon not in found:
                found.append(canon)
            s = s.replace(f" {ph} ", " ")
    for tok in re.findall(r"[a-z]+", s):
        canon = normalise_query_term(tok)
        if (tok in SINGLE_TOKENS or canon in KNOWN_VOCAB) and canon not in blocked and canon not in found:
            # Skip generic adjectives/meal words
            if canon not in {"quick", "easy", "dinner", "lunch", "breakfast", "snack", "soup", "salad", "recipe", "recipes", "meal", "food", "dish", "free"}:
                found.append(canon)
    return found

def parse_with_rules(raw_query: str) -> tuple[ParsedQuery, bool]:
    q = raw_query.strip()
    diet_tags = []
    cleaned = q

    for tag, pat in DIET_PATTERNS:
        if pat.search(cleaned):
            diet_tags.append(tag)
            cleaned = pat.sub(" ", cleaned)

    max_minutes = None
    m_time = TIME_RE.search(cleaned)
    if m_time:
        max_minutes = max(1, min(480, int(m_time.group(1))))
        cleaned = TIME_RE.sub(" ", cleaned)
    elif re.search(r"\bquick\b|\bfast\b", cleaned, re.I):
        max_minutes = 25

    exclude_ings = []
    exclude_allergens = []
    for m in NEG_RE.finditer(cleaned):
        raw_ex = m.group(1).strip().lower()
        for piece in re.split(r"\s+or\s+|\s+and\s+", raw_ex):
            piece = piece.strip()
            if not piece:
                continue
            if piece in ALLERGEN_EXPANSIONS:
                for exp in ALLERGEN_EXPANSIONS[piece]:
                    if exp not in exclude_ings:
                        exclude_ings.append(exp)
                if piece in ("dairy", "lactose"):
                    exclude_allergens.append("dairy")
                elif "nut" in piece:
                    exclude_allergens.extend(["peanut", "tree_nut"])
                elif "egg" in piece:
                    exclude_allergens.append("egg")
                elif "soy" in piece:
                    exclude_allergens.append("soy")
            else:
                c = normalise_query_term(piece)
                if c and c not in exclude_ings:
                    exclude_ings.append(c)
    cleaned = NEG_RE.sub(" ", cleaned)

    include_ings = extract_ingredients_from_text(cleaned, blocked=set(exclude_ings))
    needs_llm = bool(COMPLEX_HINT_RE.search(q)) or (not include_ings and not diet_tags and max_minutes is None and len(q.split()) >= 4)

    parsed = ParsedQuery(
        include_ingredients=include_ings,
        exclude_ingredients=exclude_ings,
        exclude_allergens=sorted(set(exclude_allergens)),
        diet_tags=diet_tags,
        max_minutes=max_minutes,
        semantic_query=q,
        parser_mode="rules",
    )
    return parsed, needs_llm


In [9]:
def parse_with_llm(raw_query: str) -> ParsedQuery:
    prompt = f"""Extract structured recipe search constraints from this user query.
Return ONLY valid JSON matching this schema (no markdown, no explanation):
{{
  "include_ingredients": ["lowercase ingredient names"],
  "exclude_ingredients": ["lowercase excluded ingredient names"],
  "diet_tags": ["subset of: gluten_free, halal_friendly, vegan, nut_free"],
  "max_minutes": integer or null,
  "semantic_query": "short clean search phrase"
}}
User query: {raw_query}"""
    messages = [{"role": "user", "content": prompt}]
    text = llm_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = llm_tokenizer([text], return_tensors="pt").to(llm_model.device)
    with torch.no_grad():
        out = llm_model.generate(**inputs, max_new_tokens=180, do_sample=False)
    gen = llm_tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    m = re.search(r"\{.*\}", gen, re.S)
    if not m:
        raise ValueError(f"LLM returned non-JSON: {gen}")
    data = json.loads(m.group(0))
    data["include_ingredients"] = [normalise_query_term(x) for x in data.get("include_ingredients", []) if x]
    ex = []
    for x in data.get("exclude_ingredients", []):
        xl = x.lower().strip()
        if xl in ALLERGEN_EXPANSIONS:
            ex.extend(ALLERGEN_EXPANSIONS[xl])
        elif xl:
            ex.append(normalise_query_term(xl))
    data["exclude_ingredients"] = list(dict.fromkeys(ex))
    data["parser_mode"] = "llm_fallback"
    return ParsedQuery(**data)

def parse_query(raw_query: str, force_llm: bool = False) -> ParsedQuery:
    parsed, needs_llm = parse_with_rules(raw_query)
    if force_llm or needs_llm:
        try:
            llm_parsed = parse_with_llm(raw_query)
            # Merge deterministic safety tags so LLM never drops an explicit diet/time constraint
            llm_parsed.diet_tags = sorted(set(llm_parsed.diet_tags) | set(parsed.diet_tags))
            if parsed.max_minutes and not llm_parsed.max_minutes:
                llm_parsed.max_minutes = parsed.max_minutes
            llm_parsed.exclude_allergens = sorted(set(parsed.exclude_allergens))
            llm_parsed.exclude_ingredients = list(dict.fromkeys(llm_parsed.exclude_ingredients + parsed.exclude_ingredients))
            return llm_parsed
        except (ValidationError, Exception):
            return parsed
    return parsed

# Quick parser verification
p1 = parse_query("rice, egg, spinach under 20 mins, gluten free, no dairy")
assert "rice" in p1.include_ingredients and "gluten_free" in p1.diet_tags and p1.max_minutes == 20 and "milk" in p1.exclude_ingredients
p2 = parse_query("something comforting with chickpeas and coconut milk, vegan, nut free, under 30 min")
assert "vegan" in p2.diet_tags and "nut_free" in p2.diet_tags and p2.max_minutes == 30
print("Parser checks PASSED! Example:", p1.model_dump())

Parser checks PASSED! Example: {'include_ingredients': ['rice', 'egg', 'spinach'], 'exclude_ingredients': ['milk', 'butter', 'cream', 'heavy cream', 'sour cream', 'cheese', 'cheddar', 'parmesan', 'mozzarella', 'feta', 'cream cheese', 'yogurt', 'greek yogurt', 'ghee', 'buttermilk'], 'exclude_allergens': ['dairy'], 'diet_tags': ['gluten_free'], 'max_minutes': 20, 'semantic_query': 'rice, egg, spinach under 20 mins, gluten free, no dairy', 'parser_mode': 'rules'}


Retrieval Variants (0–5), Coverage Scoring, Reranking & Relaxation

In [10]:
import time
import numpy as np
from qdrant_client import models

def build_qdrant_filter(parsed: ParsedQuery) -> models.Filter | None:
    must = []
    must_not = []
    for tag in parsed.diet_tags:
        must.append(models.FieldCondition(key="diet_tags", match=models.MatchValue(value=tag)))
    if parsed.max_minutes is not None:
        must.append(models.FieldCondition(key="total_minutes", range=models.Range(lte=int(parsed.max_minutes))))
    if parsed.exclude_ingredients:
        must_not.append(models.FieldCondition(key="ingredients", match=models.MatchAny(any=parsed.exclude_ingredients)))
    if parsed.exclude_allergens:
        must_not.append(models.FieldCondition(key="allergens_present", match=models.MatchAny(any=parsed.exclude_allergens)))
    if not must and not must_not:
        return None
    return models.Filter(must=must or None, must_not=must_not or None)

def satisfies_constraints(payload: dict, parsed: ParsedQuery) -> bool:
    for tag in parsed.diet_tags:
        if tag not in payload.get("diet_tags", []):
            return False
    if parsed.max_minutes is not None and payload.get("total_minutes", 9999) > parsed.max_minutes:
        return False
    p_ings = set(payload.get("ingredients", []))
    if set(parsed.exclude_ingredients) & p_ings:
        return False
    p_all = set(payload.get("allergens_present", []))
    if set(parsed.exclude_allergens) & p_all:
        return False
    return True

def compute_coverage(recipe_ings: list[str], user_ings: list[str]) -> tuple[list[str], list[str], float]:
    r_set = set(recipe_ings)
    u_set = set(user_ings)
    if not u_set:
        return ([], list(recipe_ings), 0.0)
    matched = [i for i in user_ings if i in r_set]
    missing = [i for i in recipe_ings if i not in u_set]
    user_recall = len(matched) / len(u_set)
    pantry_fit = len(matched) / max(len(r_set), 1)
    cov_score = 0.65 * user_recall + 0.35 * pantry_fit
    return matched, missing, cov_score

In [11]:
def search_recipes(parsed: ParsedQuery, mode: str = "v5_hybrid_coverage_rerank", top_k: int = 10) -> tuple[list[dict], float]:
    t0 = time.perf_counter()
    q_text = parsed.semantic_query or ", ".join(parsed.include_ingredients) or "healthy dinner recipe"
    q_filter = build_qdrant_filter(parsed)

    q_dense = dense_model.encode(q_text, normalize_embeddings=True).tolist()
    sp = next(sparse_model.query_embed(q_text))
    q_sparse = models.SparseVector(indices=sp.indices.tolist(), values=sp.values.tolist())

    if mode == "v0_naive_postfilter":
        # Naive RAG baseline: dense retrieval WITHOUT in-index filters, then post-filter top 30
        res = client.query_points(
            collection_name=COLLECTION_NAME,
            query=q_dense,
            using="dense",
            limit=30,
            with_payload=True,
        )
        points = [p for p in res.points if satisfies_constraints(p.payload, parsed)][:top_k]
    elif mode == "v0_naive_unfiltered":
        # Unfiltered dense RAG (used to measure raw constraint violation rate!)
        res = client.query_points(
            collection_name=COLLECTION_NAME,
            query=q_dense,
            using="dense",
            limit=top_k,
            with_payload=True,
        )
        points = res.points
    elif mode == "v1_in_index_dense":
        res = client.query_points(
            collection_name=COLLECTION_NAME,
            query=q_dense,
            using="dense",
            query_filter=q_filter,
            limit=top_k,
            with_payload=True,
        )
        points = res.points
    elif mode == "v2_in_index_bm25":
        res = client.query_points(
            collection_name=COLLECTION_NAME,
            query=q_sparse,
            using="bm25",
            query_filter=q_filter,
            limit=top_k,
            with_payload=True,
        )
        points = res.points
    else:
        # Hybrid Prefetch + RRF (v3, v4, v5)
        fetch_n = top_k if mode == "v3_hybrid_rrf" else 60
        res = client.query_points(
            collection_name=COLLECTION_NAME,
            prefetch=[
                models.Prefetch(query=q_dense, using="dense", filter=q_filter, limit=fetch_n),
                models.Prefetch(query=q_sparse, using="bm25", filter=q_filter, limit=fetch_n),
            ],
            query=models.FusionQuery(fusion=models.Fusion.RRF),
            limit=fetch_n,
            with_payload=True,
        )
        points = res.points

    if not points:
        return [], (time.perf_counter() - t0) * 1000

    max_s = max(p.score for p in points) or 1.0
    candidates = []
    for p in points:
        pl = p.payload
        matched, missing, cov = compute_coverage(pl["ingredients"], parsed.include_ingredients)
        rrf_norm = p.score / max_s
        rating_norm = min(pl.get("rating", 0.0) / 5.0, 1.0)
        cov_blend = (0.45 * rrf_norm + 0.45 * cov + 0.10 * rating_norm) if parsed.include_ingredients else (0.85 * rrf_norm + 0.15 * rating_norm)
        candidates.append({
            "point": p,
            "payload": pl,
            "matched": matched,
            "missing": missing,
            "coverage": cov,
            "rrf_norm": rrf_norm,
            "score": cov_blend if mode in ("v4_hybrid_coverage", "v5_hybrid_coverage_rerank") else p.score,
        })

    if mode in ("v4_hybrid_coverage", "v5_hybrid_coverage_rerank"):
        candidates.sort(key=lambda x: x["score"], reverse=True)

    if mode == "v5_hybrid_coverage_rerank":
        top_rerank = candidates[:25]
        pairs = [
            (q_text, f"{c['payload']['title']}. Ingredients: {', '.join(c['payload']['ingredients'])}")
            for c in top_rerank
        ]
        ce_raw = reranker_model.predict(pairs, batch_size=25, show_progress_bar=False)
        ce_probs = 1.0 / (1.0 + np.exp(-np.array(ce_raw)))
        for c, prob in zip(top_rerank, ce_probs):
            if parsed.include_ingredients:
                c["score"] = 0.40 * float(prob) + 0.45 * c["coverage"] + 0.15 * c["rrf_norm"]
            else:
                c["score"] = 0.75 * float(prob) + 0.25 * c["rrf_norm"]
        top_rerank.sort(key=lambda x: x["score"], reverse=True)
        candidates = top_rerank

    results = []
    for c in candidates[:top_k]:
        pl = dict(c["payload"])
        pl["matched_ingredients"] = c["matched"]
        pl["missing_ingredients"] = c["missing"]
        pl["coverage_pct"] = round(c["coverage"] * 100, 1)
        pl["final_score"] = round(float(c["score"]), 4)
        results.append(pl)

    elapsed_ms = (time.perf_counter() - t0) * 1000
    return results, elapsed_ms

In [12]:
def suggest_relaxations(parsed: ParsedQuery) -> list[str]:
    tips = []
    if parsed.max_minutes is not None and parsed.max_minutes < 60:
        p_rel = parsed.model_copy(update={"max_minutes": parsed.max_minutes + 25})
        cnt = client.count(COLLECTION_NAME, count_filter=build_qdrant_filter(p_rel), exact=False).count
        if cnt > 0:
            tips.append(f"Increase time limit from {parsed.max_minutes} min to {p_rel.max_minutes} min → **{cnt:,} matching recipes**")
    if parsed.exclude_ingredients:
        p_rel = parsed.model_copy(update={"exclude_ingredients": [], "exclude_allergens": []})
        cnt = client.count(COLLECTION_NAME, count_filter=build_qdrant_filter(p_rel), exact=False).count
        if cnt > 0:
            tips.append(f"Drop ingredient exclusions ({', '.join(parsed.exclude_ingredients[:4])}) → **{cnt:,} recipes**")
    if len(parsed.diet_tags) > 1:
        for t in parsed.diet_tags:
            p_rel = parsed.model_copy(update={"diet_tags": [x for x in parsed.diet_tags if x != t]})
            cnt = client.count(COLLECTION_NAME, count_filter=build_qdrant_filter(p_rel), exact=False).count
            if cnt > 0:
                tips.append(f"⚠️ Relax `{t}` filter (only if safe for you!) → **{cnt:,} recipes**")
    return tips